In [1]:
# Production Setup: Install necessary 3D and Vision libraries
!pip install -q open3d
!pip install -q torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.4.1+cu121.html
!pip install -q timm  # For DINOv2 weights

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 4.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 102.2 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 58.9 MB/s eta 0:00:0000:010:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 100.0 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 88.3 MB/s eta 0:00:00:00:01


In [2]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import timm
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# --- PRODUCTION CONFIGURATION ---
CONFIG = {
    "ROOT_3D": '/kaggle/input/datasets/klein2111/scannet-3d/scannet_3d/train',
    "ROOT_2D": '/kaggle/input/datasets/klein2111/scannet-2d/scannet_2d',
    "NUM_POINTS": 65536,    # Fixed point count for batching
    "BATCH_SIZE": 4,        # Fits in T4 GPU
    "NUM_CLASSES": 20,      # Target classes for the demo
    "PRETRAIN_EPOCHS": 3,   # Phase 1: Teach geometry to understand semantics
    "FINETUNE_EPOCHS": 10,  # Phase 2: Train the specific classification head
    "LR_PRETRAIN": 0.004,
    "LR_FINETUNE": 0.001,
    "DEVICE": torch.device("cuda" if torch.cuda.is_available() else "cpu")
}

print(f"Running on: {CONFIG['DEVICE']}")

Running on: cuda


In [3]:
class ScanNetProductionDataset(Dataset):
    def __init__(self, root_3d, root_2d, num_points=65536):
        self.root_3d = root_3d
        self.root_2d = root_2d
        self.num_points = num_points
        self.file_list = sorted([f for f in os.listdir(root_3d) if f.endswith('.pth')])

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        file_name = self.file_list[idx]
        scene_id = file_name.replace('_vh_clean_2.pth', '')
        
        # 1. Load 3D Data
        pc_path = os.path.join(self.root_3d, file_name)
        # weights_only=False handles the numpy/tuple structures in ScanNet
        data_3d = torch.load(pc_path, weights_only=False)
        
        # Extract Points & Labels Robustly
        if isinstance(data_3d, tuple):
            points = data_3d[0]
            # Try index 2 for labels, fallback to index 1
            labels = data_3d[2] if len(data_3d) > 2 else data_3d[1] 
        elif isinstance(data_3d, dict):
            points = data_3d['coord']
            labels = data_3d.get('semantic_gt', data_3d.get('label', np.zeros(len(points))))
        else:
            points = data_3d[:, :3]
            labels = np.zeros(len(points))
            
        points = np.array(points) if torch.is_tensor(points) else points
        labels = np.array(labels) if torch.is_tensor(labels) else labels

        # 2. Strict Shape Enforcement (Upsample/Downsample)
        if len(points) >= self.num_points:
            s_idx = np.random.choice(len(points), self.num_points, replace=False)
        else:
            s_idx = np.random.choice(len(points), self.num_points, replace=True)
            
        points = points[s_idx]
        labels = labels[s_idx]
            
        # 3. Load 2D Image (For Pre-training Phase)
        img_path = os.path.join(self.root_2d, scene_id, "color", "0.jpg")
        if os.path.exists(img_path):
            image = cv2.imread(img_path)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            image = cv2.resize(image, (518, 518))
            image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        else:
            image = torch.zeros((3, 518, 518))

        return {
            "points": torch.as_tensor(points).float(), 
            "labels": torch.as_tensor(labels).long(),
            "image": image,
            "scene_id": scene_id
        }

In [4]:
# --- 1. THE TEACHER (DINOv2) ---
def load_2d_teacher():
    model = timm.create_model('vit_small_patch14_dinov2', pretrained=True)
    for param in model.parameters():
        param.requires_grad = False
    model.eval()
    return model

# --- 2. THE STUDENT (Backbone) ---
class ConcertoStudent(nn.Module):
    def __init__(self, output_dim=384): 
        super().__init__()
        # Simulating PTv3 backbone
        self.backbone = nn.Sequential(
            nn.Linear(3, 128), nn.ReLU(),
            nn.Linear(128, 384)
        )
        self.predictor = nn.Linear(384, output_dim)

    def forward(self, x):
        features = self.backbone(x)
        return self.predictor(features)

# --- 3. LoRA LAYER (Efficiency) ---
class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank=8, alpha=16, dropout=0.1):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.linear.weight.requires_grad = False # Freeze base
        self.linear.bias.requires_grad = False
        
        self.lora_A = nn.Linear(in_features, rank, bias=False)
        self.lora_B = nn.Linear(rank, out_features, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.scaling = alpha / rank
        
        nn.init.kaiming_uniform_(self.lora_A.weight)
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x):
        return self.linear(x) + (self.lora_B(self.lora_A(self.dropout(x))) * self.scaling)

# --- 4. FINAL PRODUCTION MODEL ---
class SpatialSynergyNet(nn.Module):
    def __init__(self, pretrained_backbone, num_classes=20):
        super().__init__()
        self.backbone = pretrained_backbone
        # Freeze the backbone (it was trained in Phase 1)
        for param in self.backbone.parameters():
            param.requires_grad = False
            
        # The Head: Learns to classify using LoRA
        self.task_head = nn.Sequential(
            LoRALinear(384, 128, rank=8),
            nn.ReLU(),
            nn.Linear(128, num_classes) 
        )

    def forward(self, x):
        features = self.backbone(x) # Extract geometric-semantic features
        return self.task_head(features) # Classify

In [5]:
def run_production_pipeline():
    # 1. Initialize Data
    dataset = ScanNetProductionDataset(CONFIG['ROOT_3D'], CONFIG['ROOT_2D'])
    loader = DataLoader(dataset, batch_size=CONFIG['BATCH_SIZE'], shuffle=True)
    print(f"Data Loaded: {len(dataset)} scenes ready.")

    # 2. Setup Models
    student = ConcertoStudent().to(CONFIG['DEVICE'])
    teacher = load_2d_teacher().to(CONFIG['DEVICE'])
    
    # --- PHASE 1: PRE-TRAINING (Geometry <-> Semantics) ---
    print("\n--- PHASE 1: PRE-TRAINING (Joint SSL) ---")
    optimizer_pre = optim.AdamW(student.parameters(), lr=CONFIG['LR_PRETRAIN'])
    
    for epoch in range(CONFIG['PRETRAIN_EPOCHS']):
        student.train()
        total_loss = 0
        pbar = tqdm(loader, desc=f"Pretrain Epoch {epoch+1}")
        
        for batch in pbar:
            points = batch['points'].to(CONFIG['DEVICE'])
            images = batch['image'].to(CONFIG['DEVICE'])
            
            optimizer_pre.zero_grad()
            
            # Forward
            s_feat = student(points)       # [B, N, 384]
            with torch.no_grad():
                t_feat = teacher(images)   # [B, 384]
            
            # Synergy Loss
            s_global = s_feat.mean(dim=1) 
            loss = (1 - F.cosine_similarity(s_global, t_feat, dim=1)).mean()
            
            loss.backward()
            optimizer_pre.step()
            total_loss += loss.item()
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})
            
    # Save the base knowledge
    torch.save(student.backbone.state_dict(), "backbone_pretrained.pth")
    print("Phase 1 Complete. Backbone Saved.")

    # --- PHASE 2: FINE-TUNING (Real Labels) ---
    print("\n--- PHASE 2: FINE-TUNING (LoRA Semantic Segmentation) ---")
    
    # Initialize the final model with the pretrained backbone
    final_model = SpatialSynergyNet(student.backbone, num_classes=CONFIG['NUM_CLASSES']).to(CONFIG['DEVICE'])
    
    # Train ONLY the LoRA parameters
    trainable_params = filter(lambda p: p.requires_grad, final_model.parameters())
    optimizer_lora = optim.AdamW(trainable_params, lr=CONFIG['LR_FINETUNE'])
    
    # Loss: Ignore index 255 (Unlabeled/Noise)
    criterion = nn.CrossEntropyLoss(ignore_index=255)
    
    for epoch in range(CONFIG['FINETUNE_EPOCHS']):
        final_model.train()
        total_loss = 0
        pbar = tqdm(loader, desc=f"Finetune Epoch {epoch+1}")
        
        for batch in pbar:
            points = batch['points'].to(CONFIG['DEVICE'])
            labels = batch['labels'].to(CONFIG['DEVICE'])
            
            # Sanitize Labels (Clamp to 0-19, set rest to 255)
            labels[(labels < 0) | (labels >= CONFIG['NUM_CLASSES'])] = 255
            
            optimizer_lora.zero_grad()
            
            # Predict
            preds = final_model(points) # [B, N, 20]
            preds = preds.permute(0, 2, 1) # [B, 20, N] for CrossEntropy
            
            loss = criterion(preds, labels)
            loss.backward()
            optimizer_lora.step()
            
            total_loss += loss.item()
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})
            
        print(f"Epoch {epoch+1} Avg Loss: {total_loss/len(loader):.4f}")

    # --- FINAL SAVE ---
    # We save the state_dict. This is the standard, safest format for PyTorch.
    torch.save(final_model.state_dict(), "SpatialSynergyNet_Production.pth")
    print("\nSUCCESS: Model saved as 'SpatialSynergyNet_Production.pth'")
    print("Download this file for your local demo!")

# Run the whole thing
run_production_pipeline()

Data Loaded: 1201 scenes ready.


model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]


--- PHASE 1: PRE-TRAINING (Joint SSL) ---


Pretrain Epoch 3: 100%|██████████| 301/301 [01:44<00:00,  2.89it/s, loss=0.3134]


Phase 1 Complete. Backbone Saved.

--- PHASE 2: FINE-TUNING (LoRA Semantic Segmentation) ---


...

Epoch 8 Avg Loss: 1.6923


Finetune Epoch 9: 100%|██████████| 301/301 [00:53<00:00,  5.59it/s, loss=1.6861]


Epoch 9 Avg Loss: 1.6836


Finetune Epoch 10: 100%|██████████| 301/301 [00:53<00:00,  5.65it/s, loss=1.1545]

Epoch 10 Avg Loss: 1.6823

SUCCESS: Model saved as 'SpatialSynergyNet_Production.pth'
Download this file for your local demo!
